# Snowflake ML


<a id="what_is_snowpark"></a>
## 1. What is Snowflake ML?

Snowflake ML is the overarching name for several components that enable Machine Learning on Snowflake.

<img src="../../images/new_snowflake_ml.png" alt="Snowpark ML" style="width:100%;display:block;margin-left:0%;" />


Snowflake ML components help to streamline the ML lifecycle, as shown here.


<img src="../../images/snowflake_ml_process.png" alt="Snowpark ML" style="width:70%;display:block;margin-left:10%;" />




Snowpark ML is a component of Snowflake ML. It is a Python library with several APIs that allow you to use:
- ML Modeling
- Feature Store
- Model Registry

With Snowpark ML Data Scientist and ML engineers can use familiar Python frameworks to do feature engineering and model training for models that can be managed **entirely in Snowflake without any data movement**, siloes or governance trade offs. 




<img src="../../images/snowflake_ml_components.png" alt="Snowpark ML" style="width:70%;display:block;margin-left:10%;" />


By letting you perform these tasks in a Snowflake Python application, Snowpark ML provides the following advantages:

- Transform your data and train your models without moving your data out of Snowflake and without having to define and deploy stored procedures that package scikit-learn, xgboost, or lightgbm code.

- Work with APIs similar to those you're already familiar with, such as scikit-learn.

- Keep your ML pipeline running within Snowflake's security and governance frameworks.

- Take advantage of the performance and scalability of Snowflake's data warehouses.


<img src="../../images/Snowpark_ML_Arch.png" alt="Snowparks ML" style="width:70%;display:block;margin-left:10%;" />

## 1a.Snowpark ML Modeling API
Snowpark ML Modeling supports data preprocessing, feature engineering, and model training in Snowflake using popular machine learning frameworks.

The Snowpark ML Modeling API consists of the following:

- Feature Engineering and Preprocessing: Improve performance and scalability with distributed execution for common scikit-learn preprocessing functions. 

- Model Training: Simplify model training for scikit-learn and xgboost models.

Now you can run popular frameworks like scikit-learn and xgboost natively in Snowflake without data movement.

**Note**: Data Preprocessing includes the steps we need to follow to transform or encode data so that it may be easily parsed by the machine. 

<img src="../../images/Snowpark_modeling_api_2.png" alt="MLAPIQuery" style="width:65%;display:block;margin-left:10%;" />
 

## 1b. Snowflake Model Registry API 

Snowflake ML Model Registry allows you to manage models regardless of origin!


**We will discuss the Model Registry later in this course.**

To give you an overview of where Snowflake ML lands in Snowflake:

<img src="../../images/Snowflake_Machine_Learning.png" alt="SFML" style="width:85%;display:block;margin-left:10%;" />

## 2. Snowpark ML Modeling

Let's talk about the Snowpark ML Modeling.

With Snowpark ML Modeling, you can do things like:
- Preprocessing
- Pipelines
- Model training
- Distributed Hyperparameter Optimization
- Deplying models
- Running models

These things can be done in a similar way as other machine learning libraries that you might be familiar with.

## 2a. Preprocessing

Preprocessing is a component of the Snowpark ML Modeling API and can be found under `snowflake.ml.modeling.preprocessing`.

### Distributed Preprocessing
Many of the data preprocessing and transformation functions in Snowpark ML are implemented using Snowflake’s distributed execution engine, which provides significant performance benefit compared to single-node execution (that is, stored procedures).

The chart below shows illustrative performance numbers on large public datasets, running in a medium Snowpark-optimized warehouse, comparing scikit-learn running in stored procedures to Snowpark ML’s distributed implementations. In many scenarios, your code can run 25 to 50 times faster when using Snowpark ML Modeling.

<img src="../../images/snowpark_ml_distributed_performance.png" alt="MLAPIPerformance" style="width:85%;display:block;margin-left:10%;" />


### Fit and Transform

In Snowpark the `fit` method accepts a Snowpark or Pandas DataFrame and returns a fitter transformer

- Snowpark dataframe: the fitting is distributed and uses the SQL engine
- Pandas Dataframe: the fitting is done locally, similar to scikit-learn

The `transform` method of a Snowpark ML preprocessing transformer accepts a Snowpark or Pandas DataFrame, transforms the dataset, and returns a transformed dataset.

- Snowpark dataframe: the distributed transformation lazely uses the SQL engine
- Pandas Dataframe: transformed locally, similar to scikit-learn

**Note**: Certain complex transformations involve execution. This includes transformers that require temporary state tables (such as OneHotEncoder and OrdinalEncoder) during transformation. 

### MyNote (well Claude's note) re Transformers vs Pipelines

  Transformers (e.g. MinMaxScaler, OneHotEncoder, Binarizer) are individual steps — each one does a single transformation. They have:
  - .fit() — learns parameters from data (e.g. min/max values)
  - .transform() — applies the learned transformation to data
  - .fit_transform() — convenience method combining both 
  
  Pipelines are containers that chain transformers and estimators together in sequence. A Pipeline:
  - Holds an ordered list of steps (transformers + optionally a final estimator like KMeans)
  - Calls .fit() on each step in order during training
  - Passes each step's output as the next step's input
  - Exposes a single .fit() / .predict() interface for the whole chain
  
  So the relationship is:

  Pipeline  
  ├── Step 1: MinMaxScaler (transformer)
  ├── Step 2: PCA (transformer)
  └── Step 3: KMeans (estimator)

  When to use which:
  - Use a transformer alone when you just need one transformation and want to apply it independently
  - Use a Pipeline when you have multiple steps that must be applied in sequence, especially for training/inference where you want to ensure the same sequence runs consistently every time
  
  In the feature engineering notebook you worked on earlier, MinMaxScaler and KMeans were individual transformers/estimators wrapped together in a Pipeline — exactly this pattern.


### Snowpark Pipeline

A series of transformations is common. However, scikit-learn pipelines are not supported. Snowpark modeling has the `snowflake.ml.modeling.pipeline` class which works the same as the scikit-learn version. 


### Distributed Hyperparameter Optimization

The Snowpark ML library provides distributed implementations of the scikit-learn `GridSearchCV` and `RandomizedSearchCV` APIs to enable efficient hyperparameter tuning on both single-node and multiple-node warehouses.

**Note**: Snowpark ML enables distributed hyperparameter optimization by default

<img src="../../images/snowpark_ml_distributed_architecture.png" alt="MLAPIPerformance" style="width:85%;display:block;margin-left:10%;" />

On a single node warehouse, such as a XS warehouse or a Snowpark-optimized warehouse, you can get out of memory exeptions. If that is the case, reduce parallelism by specifying n_jobs parameter, which by default uses all cores. 

For multi-node warehouses, distribution across all cores and all nodes is done automatically. 

### Model training

The Snowpark ML Modeling API provides wrappers for underlying scikit-learn, xgboost, and lightgbm classes, the majority of which are executed as stored procedures (running on a single warehouse node) in the virtual warehouse.

### Model inference

The result of a training a model is a Python Snowpark ML model object. You can use the trained model to make predictions by calling the model’s predict method. 

*This creates a temporary user-defined function to run the model in your Snowflake virtual warehouse and is automatically deleted at the end of your session*

You can also manually create the UDF to make it permanent. 

### Using the model object

Snowpark ML models can be “unwrapped,” that is, converted to the underlying third-party models, with the following methods (depending on the library):

`to_sklearn`

`to_xgboost`

`to_lightgbm`

## Let's take a look a the API

At its core it is `snowflake.ml.modeling`

Under modeling we have the following categories:

- `snowflake.ml.modeling.calibration`
- `snowflake.ml.modeling.cluster`
- `snowflake.ml.modeling.compose`
- `snowflake.ml.modeling.covariance`
- `snowflake.ml.modeling.decomposition`
- `snowflake.ml.modeling.discriminant_analysis`
- `snowflake.ml.modeling.ensemble`
- `snowflake.ml.modeling.feature_selection`
- `snowflake.ml.modeling.gaussian_process`
- `snowflake.ml.modeling.impute`
- `snowflake.ml.modeling.kernel_approximation`
- `snowflake.ml.modeling.lightgbm`
- `snowflake.ml.modeling.linear_model`
- `snowflake.ml.modeling.manifold`
- `snowflake.ml.modeling.metrics`
- `snowflake.ml.modeling.mixture`
- `snowflake.ml.modeling.model_selection`
- `snowflake.ml.modeling.multiclass`
- `snowflake.ml.modeling.naive_bayes`
- `snowflake.ml.modeling.neighbors`
- `snowflake.ml.modeling.neural_network`
- `snowflake.ml.modeling.pipeline`
- `snowflake.ml.modeling.preprocessing`
- `snowflake.ml.modeling.semi_supervised`
- `snowflake.ml.modeling.svm`
- `snowflake.ml.modeling.tree`
- `snowflake.ml.modeling.xgboost`
- `snowflake.ml.fileset`

Now, let's take a look at some useful model categories.


### Preprocessing & Pipeline

In the demo later in this notebook we'll show some preprocessing steps, such as `OneHotEncoder.`

Each preprocessor comes with its own set of methods with at least `fit()` and `transform()`
- `fit()` returns self
- `transform()` returns snowpark dataframe

`snowflake.ml.modeling.pipeline.Pipeline` uses the same base: `basetransformer`. It also comes with its own set of methods and has a similar `fit()` and `transform()`

### Fileset

Fileset comes with 2 classes
- `sfcfs.SFFileSystem`

A filesystem that allows user to access Snowflake stages and stage files with valid Snowflake locations.

- `fileset.FileSet`

A FileSet represents an immutable snapshot of the result of a query in the form of files.

> **&#128221; Note:** See documentation for further details:
> - [snowflake.ml.fileset](https://docs.snowflake.com/en/developer-guide/snowpark-ml/reference/latest/fileset)


### Feature_selection

`snowflake.ml.modeling.feature_selection` gives a range of classes to perform feature selection, such as `GenericUnivariateSelect()`.

These classes use the `BaseTransformer` so it comes again with `fit()` and `transform()`



### Metrics

Metrics consists of functions to calculate metrics, such as accuracy using `accuracy_score()`

Which in this case returns a float. 

### Model_selection

`snowflake.ml.modeling.model_selection` consist of 2 classes:

- `GridSearchCV`
- `RandomizedSearchCV`

both using the `BaseTransformer` base

#  Time for a demo (in the next notebook)!